In [ ]:
import pandas as pd
from entsoe import EntsoePandasClient
import requests

client = EntsoePandasClient(api_key='API-KEY')

start = pd.Timestamp('20240112', tz='Europe/Berlin')
#end = pd.Timestamp('20251231', tz='Europe/Berlin')
end = pd.Timestamp.now(tz='Europe/Berlin') + pd.Timedelta(days=1) # + tomorrow for prediction at 27th january
country_code = 'DE_LU'

# takes the sequence 1 prices from entsoe since it matches the data from smard
print("Fetching day ahead prices...")
prices = client.query_day_ahead_prices(country_code, start=start, end=end)

print("Fetching load forecast...")
load_forecast = client.query_load_forecast(country_code, start=start, end=end)

print("Fetching wind and solar forecast...")
wind_solar_forecast = client.query_wind_and_solar_forecast(country_code, start=start, end=end)

# aggregation
df = pd.DataFrame()
df['Price_DayAhead'] = prices
df['Load_Forecast'] = load_forecast

wind_solar_forecast.columns = wind_solar_forecast.columns.str.replace(' ', '_')

df = df.join(wind_solar_forecast)

print(df.head())
df.to_csv('entsoe_forecasts.csv')




In [ ]:

import pandas as pd
import requests
from datetime import date, timedelta

cities = {
    "Berlin": {"lat": 52.52, "lon": 13.41},
    "Hamburg": {"lat": 53.55, "lon": 9.99},
    "Köln": {"lat": 50.93, "lon": 6.95},
    "München": {"lat": 48.13, "lon": 11.58},
    "Leipzig": {"lat": 51.33, "lon": 12.37},
    "Frankfurt": {"lat": 50.11, "lon": 8.68}
}

yesterday = (date.today() - timedelta(days=1)).isoformat()
tomorrow = (date.today() + timedelta(days=1)).isoformat()

def get_weather_forecasts():
    all_city_data = []
    
    for name, coord in cities.items():
        #historal forecast and forecast are split in the open-meteo API
        url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
        params = {
            "latitude": coord["lat"],
            "longitude": coord["lon"],
            "start_date": "2024-01-12",
            "end_date": yesterday,
            "hourly": "temperature_2m,windspeed_100m,shortwave_radiation",
            "timezone": "Europe/Berlin"
        }        
        response = requests.get(url, params=params).json()
        hourly = response.get('hourly', {})
        
        df_city = pd.DataFrame({
            "timestamp": pd.to_datetime(hourly.get('time')),
            f"temp_{name}": hourly.get('temperature_2m'),
            f"wind_{name}": hourly.get('windspeed_100m'),
            f"solar_{name}": hourly.get('shortwave_radiation')
        })
        df_city.set_index("timestamp", inplace=True)
        all_city_data.append(df_city)
        
        url_forecast = "https://api.open-meteo.com/v1/forecast"
        params_forecast = {
            "latitude": coord["lat"],
            "longitude": coord["lon"],
            "hourly": "temperature_2m,windspeed_100m,shortwave_radiation",
            "forecast_days": 2,
            "timezone": "Europe/Berlin"
        }
        response_forecast = requests.get(url_forecast, params=params_forecast).json()
        hourly_forecast = response_forecast.get('hourly', {})
        
        df_city_forecast = pd.DataFrame({
            "timestamp": pd.to_datetime(hourly_forecast.get('time')),
            f"temp_{name}": hourly_forecast.get('temperature_2m'),
            f"wind_{name}": hourly_forecast.get('windspeed_100m'),
            f"solar_{name}": hourly_forecast.get('shortwave_radiation')
        })
        df_city_forecast.set_index("timestamp", inplace=True)
        all_city_data.append(df_city_forecast)

    df_combined = pd.concat(all_city_data, axis=1)
    df_combined = df_combined.groupby(level=0, axis=1).first()
    
    df_final = pd.DataFrame(index=df_combined.index)
    df_final['Weather_Temp_Forecast'] = df_combined[[col for col in df_combined.columns if 'temp' in col]].mean(axis=1)
    df_final['Weather_Wind_Forecast'] = df_combined[[col for col in df_combined.columns if 'wind' in col]].mean(axis=1)
    df_final['Weather_Solar_Forecast'] = df_combined[[col for col in df_combined.columns if 'solar' in col]].mean(axis=1)
    
    return df_final

weather_df = get_weather_forecasts()
weather_df = weather_df.round(1) # part of exercise2
weather_df.to_csv("weather_forecasts_aggregated.csv")
